What is Human-in-the-Loop?
In real-world applications, you cannot let an AI agent have 100% autonomous control over everything. If an agent decides to delete a database, send an email to a premium client, or process a $10,000 credit card transaction, you need a human gatekeeper to review and click "Approve" first.

Because LangGraph saves a snapshot of your state to your SQLite database after every node, it allows you to deliberately pause the graph right before a critical node executes.

The graph completely freezes, saves its place on disk, exits the CPU memory, and waits. Hours or days later, a human can inspect the state, alter it if necessary, and tell the graph to resume from the exact millimeter it paused.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. State Definition
class PaymentState(TypedDict):
    amount: float
    account_id: str
    status: str

# 2. Nodes
def order_processor_node(state: PaymentState):
    print("📦 [Order Processor] Checking item availability...")
    return {"status": "pending_approval"}

def transfer_funds_node(state: PaymentState):
    print(f"💳 [Bank Gateway] SUCCESS! Charged ${state['amount']} to account {state['account_id']}")
    return {"status": "transaction_complete"}

# 3. Build the Graph Map
builder = StateGraph(PaymentState)
builder.add_node("processor", order_processor_node)
builder.add_node("charger", transfer_funds_node)

builder.add_edge(START, "processor")
builder.add_edge("processor", "charger")  # The road goes straight through...
builder.add_edge("charger", END)

# =========================================================
# THE HUMAN-IN-THE-LOOP MAGIC
# =========================================================
memory = MemorySaver()

# We compile the graph, but draw a police checkpoint BEFORE "charger"
app = builder.compile(
    checkpointer=memory,
    interrupt_before=["charger"]  # <-- Halt execution right before this node fires!
)
print("🛑 Secure Graph Compiled with Human-in-the-Loop Gates!")

🛑 Secure Graph Compiled with Human-in-the-Loop Gates!


In [4]:
config = {"configurable": {"thread_id": "tx_999"}}
initial_order = {"amount": 4500.00, "account_id": "acc_abhishek", "status": "initiated"}

print("--- PHASE 1: User Clicks Buy ---")
current_state = app.invoke(initial_order, config=config)

# Check where the graph stopped
snapshot = app.get_state(config)
print(f"\nIs the graph waiting for someone? {len(snapshot.next) > 0}")
print(f"Next Node in Line to execute: {snapshot.next}")
print(f"Current Database Status Value: '{snapshot.values['status']}'")

--- PHASE 1: User Clicks Buy ---
📦 [Order Processor] Checking item availability...

Is the graph waiting for someone? True
Next Node in Line to execute: ('charger',)
Current Database Status Value: 'pending_approval'


In [5]:
print("\n--- PHASE 2: Admin Reviews and Approves ---")

# To resume, we pass None as the state object. 
# LangGraph wakes up on thread tx_999, sees it was waiting at the 'charger' gate, and steps on the gas.
final_state = app.invoke(None, config=config)

print(f"\nFinal State recorded in DB: {final_state['status']}")


--- PHASE 2: Admin Reviews and Approves ---
💳 [Bank Gateway] SUCCESS! Charged $4500.0 to account acc_abhishek

Final State recorded in DB: transaction_complete
